# 01 - 数据库设计与初始化

本 Notebook 完成以下任务：
1. 数据库表结构设计说明
2. 执行 DDL 创建全部 14 张表
3. 验证表创建结果

---

## 数据库设计方案

### 分层架构

| 层级 | 表 | 更新频率 |
|------|-----|----------|
| **基础数据** | trade_calendar, stock_info, stock_pool, etf_info | 日/按需 |
| **行情数据** | stock_daily, etf_daily, index_daily | 每日 |
| **财务数据** | stock_fundamental (每日估值), stock_fina_indicator (季报) | 每日/季度 |
| **资金/事件** | stock_cashflow, stock_holder_trade, stock_holder_count | 每日/不定期 |
| **市场数据** | stock_margin | 每日 |
| **ETF 扩展** | etf_holding | 季度 |

### 主键设计

- 行情类表：`(code, trade_date)` 联合主键，天然去重
- 基础信息表：`ts_code` 主键
- 事件类表：自增 ID + UNIQUE 约束

### 增量更新基础

所有行情表通过 `trade_calendar` 对比 `trade_date` 定位缺失日期，实现精确增量采集。

## 1. 创建全部表

In [ ]:
from invest_model.db import get_engine
from invest_model.models.ddl import create_all_tables, ALL_TABLES

engine = get_engine()

created = create_all_tables(engine)
print(f"共创建/确认 {len(created)} 张表：")
for name in created:
    print(f"  - {name}")

## 2. 验证表结构

In [ ]:
from invest_model.repositories.base import BaseRepository
import pandas as pd

repo = BaseRepository(engine)

# 查看所有表
tables_df = repo.read_sql(
    "SELECT table_name, table_rows, table_comment "
    "FROM information_schema.tables "
    "WHERE table_schema = DATABASE() ORDER BY table_name"
)
tables_df.columns = tables_df.columns.str.lower()
print(f"数据库中共 {len(tables_df)} 张表：\n")
print(tables_df.to_string(index=False))

In [ ]:
# 检查缺失的表
existing = set(tables_df["table_name"].str.lower())
expected = set(ALL_TABLES)
missing = expected - existing
if missing:
    print(f"缺失的表: {missing}")
else:
    print("所有表均已创建")

In [ ]:
# 查看 stock_daily 表结构示例
cols = repo.read_sql("SHOW COLUMNS FROM stock_daily")
print("stock_daily 表结构：")
print(cols.to_string(index=False))

## 完成

14 张表已全部创建完毕，继续 `02_stock_pool.ipynb` 管理股票池。